# DiffDock検証の解析ノート

このフォルダの `README.md`（DiffDock検証レポート）に載せた数値を計算するノート。学習はやり直さず、保存済みのファイルだけを読み込む。

| セル | 内容 | レポートの節 |
|---|---|---|
| 1 | GNNのVal評価指標 | 1.1 |
| 2, 2b | 検証対象の選定条件・Val内順位・選定規則 | 4.1〜4.2 |
| 3 | Tanimoto類似度 | 4.3 |
| 4 | 立体異性体がGNNに同じ入力として見えることの確認 | 6.3 |
| 5 | DiffDockポーズの重心ズレ | 5.3 |

**読み込むファイル**: リポジトリの一番上にある `val_predictions.csv`、`chembl3_output_clean.csv`、`app_artifacts/` と、このフォルダのDiffDock出力。セル4のみ、学習ノートの特徴量セルで作る `brd4_cpi_dataset.pt` が必要。


In [1]:
# ===== 0. 準備: このノートは diffdock/ フォルダの中にある =====
import sys
ROOT = ".." #リポジトリの一番上(学習ノートが書き出したファイルがある場所)
sys.path.append(ROOT) #model_utils.py を import するため

In [2]:
# ===== 1. GNNのVal評価指標 =====
# レポート1.1節の表(Pearson / R² / MAE / RMSE)を、学習ノートのVal抽出セルが保存した予測値から計算する。
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

val_df = pd.read_csv(f"{ROOT}/val_predictions.csv")
y_true = val_df["actual_pIC50"].values #実測
y_pred = val_df["pred_pIC50"].values #予測

r, _ = pearsonr(y_pred, y_true)
mse = mean_squared_error(y_true, y_pred)

print(f"件数: {len(val_df)}")
print(f"Pearson相関係数: {r:.3f}")
print(f"R² (Pearsonの二乗。学習ノートの評価セルと同じ定義): {r**2:.3f}")
print(f"R² (決定係数 r2_score。ランダムフォレストのセルと同じ定義): {r2_score(y_true, y_pred):.3f}")
print(f"MSE: {mse:.4f}  ← 学習ログの最良Val Loss(0.3655)と一致することの確認")
print(f"RMSE: {np.sqrt(mse):.3f}")
print(f"MAE: {mean_absolute_error(y_true, y_pred):.3f}")

件数: 1336
Pearson相関係数: 0.841
R² (Pearsonの二乗。学習ノートの評価セルと同じ定義): 0.707
R² (決定係数 r2_score。ランダムフォレストのセルと同じ定義): 0.705
MSE: 0.3655  ← 学習ログの最良Val Loss(0.3655)と一致することの確認
RMSE: 0.605
MAE: 0.447


In [3]:
# ===== 2. DiffDock検証対象の選定 =====
# レポート4.1〜4.2節(選定条件、Val内順位、予測2位が対象外になる理由、過大評価の倍率)。
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

val_df = pd.read_csv(f"{ROOT}/val_predictions.csv")

# -----1. 予測pIC50の高い順に並べて、Val内順位をつける-----
val_df = val_df.sort_values("pred_pIC50", ascending=False).reset_index(drop=True)
val_df["rank"] = val_df.index + 1 #1位から数える

# -----2. 選定条件に使う値を、Val全件について計算-----
def selection_props(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return {
        "heavy_atoms": mol.GetNumHeavyAtoms(), #重原子数(水素・重水素は含まない)
        "rot_bonds": Descriptors.NumRotatableBonds(mol), #回転可能結合数(RDKit標準の定義)
        "MolWt": Descriptors.MolWt(mol), #分子量
        "n_stereo": len(Chem.FindMolChiralCenters(mol, includeUnassigned=True, useLegacyImplementation=False)), #立体中心の数(未指定のものも数える)
        "has_D": any(atom.GetAtomicNum() == 1 and atom.GetIsotope() == 2 for atom in mol.GetAtoms()), #重水素標識の有無
    }

props = pd.DataFrame(val_df["smiles"].apply(selection_props).tolist())
val_df = pd.concat([val_df, props], axis=1)

# -----3. 選定条件(4.1節)で絞り込む-----
val_df["pass"] = (
    val_df["heavy_atoms"].between(28, 48) & #重原子28〜48
    val_df["rot_bonds"].between(3, 11) & #回転可能結合3〜11
    (val_df["n_stereo"] == 0) & #立体中心なし
    (~val_df["has_D"]) #重水素なし
)
print(f"Val全体: {len(val_df)}件 → 選定条件を満たす: {val_df['pass'].sum()}件")

cols = ["rank", "pred_pIC50", "actual_pIC50", "heavy_atoms", "rot_bonds", "MolWt", "n_stereo", "has_D", "pass"]

# -----4. 予測上位5件(予測2位が選定条件を満たさないことの確認)-----
print("\n--- 予測上位5件 ---")
print(val_df[cols].head(5).to_string(index=False, float_format="%.2f"))

# -----5. DiffDockに渡した3化合物の位置-----
# SMILESはColabノートの入力CSVと同じもの
diffdock_smiles = {
    "top_pred":    "CCS(=O)(=O)Cc1ccc(Oc2ccc(F)cc2F)c(-c2cn(C)c(=O)c3cc(-c4cc(C)c(OCCO)c(C)c4)oc23)c1",
    "fp_highpred": "CCS(=O)(=O)Nc1cc(-c2cc(NCc3ccc(N(C)C)cc3)c(=O)n(C)c2)cc2c1ccn2C(C)(c1ccccn1)c1ccccn1",
    "neg_true":    "Cc1cc(-c2nc3sc4c(c3c(=O)[nH]2)CCN(C)C4)cc(C)c1OCc1cccc(C(=O)Nc2ccccc2N)c1",
}

print("\n--- DiffDockに渡した3化合物 ---")
for label, smi in diffdock_smiles.items():
    can = Chem.MolToSmiles(Chem.MolFromSmiles(smi)) #val_df側と同じ書き方(カノニカル)にそろえて照合
    hit = val_df[val_df["smiles"] == can]
    if len(hit) != 1:
        print(f"{label}: ★Val内で{len(hit)}件ヒット(1件のはず)")
        continue
    row = hit.iloc[0]
    fold = 10 ** (row["pred_pIC50"] - row["actual_pIC50"]) #IC50で見た予測と実測の倍率
    print(f"{label:<12} 順位={row['rank']:>4}  予測={row['pred_pIC50']:.2f}  実測={row['actual_pIC50']:.2f}  "
          f"重原子={row['heavy_atoms']}  回転結合={row['rot_bonds']}  条件={'OK' if row['pass'] else 'NG'}  "
          f"倍率={fold:.1f}倍")

Val全体: 1336件 → 選定条件を満たす: 461件

--- 予測上位5件 ---
 rank  pred_pIC50  actual_pIC50  heavy_atoms  rot_bonds  MolWt  n_stereo  has_D  pass
    1        9.29          9.37           44         10 623.67         0  False  True
    2        9.18          5.28           61         22 852.86         3  False False
    3        9.15          9.23           45         10 637.70         0  False  True
    4        9.14          8.66           38          5 533.09         1   True False
    5        9.12          8.93           38          5 516.64         1   True False

--- DiffDockに渡した3化合物 ---
top_pred     順位=   1  予測=9.29  実測=9.37  重原子=44  回転結合=10  条件=OK  倍率=0.8倍
fp_highpred  順位=  59  予測=8.14  実測=5.76  重原子=48  回転結合=11  条件=OK  倍率=243.0倍
neg_true     順位=1127  予測=5.46  実測=5.46  重原子=41  回転結合=6  条件=OK  倍率=1.0倍


In [4]:
# ===== 2b. 対照化合物(fp_highpred / neg_true)の選定規則 =====
# 2の val_df と cols をそのまま使う(2を実行した直後に実行)
cand = val_df[val_df["pass"]] #選定条件を満たす461件

# (a) 予測高・実測低: 実測6未満の候補を、予測の高い順に並べる → 1行目がfp_highpredなら「実測6未満で予測が最大」の規則
print("--- (a) 実測 < 6 の候補を、予測の高い順に ---")
print(cand[cand["actual_pIC50"] < 6][cols + ["abs_error"]].head(5).to_string(index=False, float_format="%.3f"))

# (b) 予測低・実測低: 予測6未満の候補を、誤差の小さい順に並べる → 1行目がneg_trueなら「予測6未満で誤差が最小」の規則
print("\n--- (b) 予測 < 6 の候補を、誤差の小さい順に ---")
print(cand[cand["pred_pIC50"] < 6].sort_values("abs_error")[cols + ["abs_error"]].head(5).to_string(index=False, float_format="%.3f"))

--- (a) 実測 < 6 の候補を、予測の高い順に ---
 rank  pred_pIC50  actual_pIC50  heavy_atoms  rot_bonds   MolWt  n_stereo  has_D  pass  abs_error
   59       8.143         5.758           48         11 661.832         0  False  True      2.386
  240       7.556         5.688           35          6 498.551         0  False  True      1.867
  278       7.484         5.643           38          6 505.626         0  False  True      1.841
  455       7.100         5.300           39          6 543.653         0  False  True      1.800
  526       6.977         5.658           32          8 491.660         0  False  True      1.320

--- (b) 予測 < 6 の候補を、誤差の小さい順に ---
 rank  pred_pIC50  actual_pIC50  heavy_atoms  rot_bonds   MolWt  n_stereo  has_D  pass  abs_error
 1127       5.455         5.458           41          6 565.699         0  False  True      0.003
 1080       5.596         5.600           30          3 440.550         0  False  True      0.004
 1104       5.543         5.552           29        

In [5]:
# ===== 3. Tanimoto類似度 =====
# レポート4.3節(JQ1とtop_predの類似度、訓練データとの最大類似度、top_predの類縁体)。
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from sklearn.model_selection import train_test_split

# -----1. Train/Valの分割を、学習ノートと同じ条件で作る-----
df_clean = pd.read_csv(f"{ROOT}/chembl3_output_clean.csv")
df_valid = df_clean[df_clean["canonicalized_smiles"].apply(lambda s: Chem.MolFromSmiles(s) is not None)].reset_index(drop=True) #Val抽出セルのvalid_smilesと同じ絞り込み
indices = list(range(len(df_valid)))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)
train_df = df_valid.iloc[train_idx]

# 分割の確認: Val側のSMILESの並びが、val_predictions.csv と完全に一致するはず
val_check = pd.read_csv(f"{ROOT}/val_predictions.csv")
assert list(df_valid.iloc[val_idx]["canonicalized_smiles"]) == list(val_check["smiles"]), "分割が一致しません"
print(f"Train: {len(train_df)}件 / Val: {len(val_idx)}件（val_predictions.csvと一致）")

# -----2. Morganフィンガープリント(半径2、2048ビット)-----
# ランダムフォレストのセルの GetMorganFingerprintAsBitVect と同じビット列になる(新しい書き方のため警告が出ない)
fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
def to_fp(smiles):
    return fpgen.GetFingerprint(Chem.MolFromSmiles(smiles))

train_fps = [to_fp(s) for s in train_df["canonicalized_smiles"]]

jq1_smiles = "CC1=C(C)C2=C(S1)N1C(C)=NN=C1[C@H](CC(=O)OC(C)(C)C)N=C2C1=CC=C(Cl)C=C1" #(S)-JQ1。Morganは立体を区別しないので(R)体でも同じ値
top_smiles = "CCS(=O)(=O)Cc1ccc(Oc2ccc(F)cc2F)c(-c2cn(C)c(=O)c3cc(-c4cc(C)c(OCCO)c(C)c4)oc23)c1"
jq1_fp = to_fp(jq1_smiles)
top_fp = to_fp(top_smiles)

# -----3. 類似度の計算-----
print(f"\nJQ1 vs top_pred: {DataStructs.TanimotoSimilarity(jq1_fp, top_fp):.3f}")

train_sims_top = DataStructs.BulkTanimotoSimilarity(top_fp, train_fps) #top_predと訓練データ全件との類似度
train_sims_jq1 = DataStructs.BulkTanimotoSimilarity(jq1_fp, train_fps) #JQ1と訓練データ全件との類似度
print(f"top_pred vs 訓練データ(最大): {max(train_sims_top):.3f}")
print(f"JQ1 vs 訓練データ(最大): {max(train_sims_jq1):.3f}")

# -----4. top_predに類似度0.7以上の訓練化合物と、その実測活性-----
near = train_df.assign(tanimoto=train_sims_top)
near = near[near["tanimoto"] >= 0.7].sort_values("tanimoto", ascending=False)
print(f"\ntop_predに類似度0.7以上の訓練化合物: {len(near)}件")
print(near[["tanimoto", "pIC50"]].to_string(index=False, float_format="%.3f"))

Train: 5341件 / Val: 1336件（val_predictions.csvと一致）

JQ1 vs top_pred: 0.113
top_pred vs 訓練データ(最大): 0.846
JQ1 vs 訓練データ(最大): 0.903

top_predに類似度0.7以上の訓練化合物: 9件
 tanimoto  pIC50
    0.846  8.668
    0.765  9.237
    0.759  9.075
    0.759  9.107
    0.753  9.264
    0.726  8.940
    0.723  8.890
    0.720  9.182
    0.709  9.000


In [6]:
# ===== 4. (S)-JQ1と(R)-JQ1が、GNNには同じ入力に見えることの確認 =====
# レポート6.3節「立体化学の非認識」。
# 定義はアプリと同じ model_utils.py から import する。brd4_cpi_dataset.pt が必要(学習ノートの特徴量セルで作成)。
import torch
import pandas as pd
from rdkit import Chem
from torch_geometric.loader import DataLoader

from model_utils import smiles_to_graph, CPIRegressor

# protein_tensorは、学習に使ったものと確実に同じにするため brd4_cpi_dataset.pt から取り出す
dataset = torch.load(f"{ROOT}/brd4_cpi_dataset.pt")
protein_tensor = dataset[0].protein_feat.squeeze(0) #[1, 125] → [125]

# 確認: model_utilsのsmiles_to_graphが、学習データと同じグラフを作るか(先頭100件)
df_clean = pd.read_csv(f"{ROOT}/chembl3_output_clean.csv")
for i in range(100):
    g = smiles_to_graph(df_clean["canonicalized_smiles"].iloc[i])
    assert all(torch.equal(g[k], dataset[i][k]) for k in ["x", "edge_index", "edge_type", "global_feat"]), f"{i}件目が一致しません"
print("smiles_to_graph: 学習データ先頭100件と一致")

jq1_S = "CC1=C(C)C2=C(S1)N1C(C)=NN=C1[C@H](CC(=O)OC(C)(C)C)N=C2C1=CC=C(Cl)C=C1"  #活性体
jq1_R = "CC1=C(C)C2=C(S1)N1C(C)=NN=C1[C@@H](CC(=O)OC(C)(C)C)N=C2C1=CC=C(Cl)C=C1" #不活性体(陰性対照)

# -----1. パイプラインと同じく、カノニカルSMILESにそろえる。立体の情報(@/@@)はSMILES上には残っている-----
can_S = Chem.MolToSmiles(Chem.MolFromSmiles(jq1_S))
can_R = Chem.MolToSmiles(Chem.MolFromSmiles(jq1_R))
for name, smi in [("S体", can_S), ("R体", can_R)]:
    print(f"{name}: CIP={Chem.FindMolChiralCenters(Chem.MolFromSmiles(smi), useLegacyImplementation=False)}  {smi}")
print(f"SMILES文字列として同じか: {can_S == can_R}")

# -----2. グラフ化した結果を、項目ごとに比較-----
g_S = smiles_to_graph(can_S)
g_R = smiles_to_graph(can_R)
print()
for key in ["x", "edge_index", "edge_type", "global_feat"]:
    print(f"{key:<12} 完全一致: {torch.equal(g_S[key], g_R[key])}")

# -----3. 学習済みGNNの予測値(学習ノートのVal抽出セルと同じ手順)-----
# top_predは、予測値が val_predictions.csv と一致するかの確認用
top_smiles = "CCS(=O)(=O)Cc1ccc(Oc2ccc(F)cc2F)c(-c2cn(C)c(=O)c3cc(-c4cc(C)c(OCCO)c(C)c4)oc23)c1"
g_top = smiles_to_graph(Chem.MolToSmiles(Chem.MolFromSmiles(top_smiles)))

scaler = torch.load(f"{ROOT}/app_artifacts/global_scaler.pt") #アプリ用の標準化パラメータ(学習時の統計量と同一)
for g in [g_S, g_R, g_top]:
    g.global_feat = (g.global_feat - scaler["mean"]) / scaler["std"] #アプリ用のファイルはキー名が mean / std
    g.protein_feat = protein_tensor.unsqueeze(0) #[125] → [1, 125]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_check = CPIRegressor().to(device) #学習済みのmodelを上書きしないよう別名にする
model_check.load_state_dict(torch.load(f"{ROOT}/app_artifacts/best_model.pt", map_location=device)) #アプリで使っている学習済みモデル
model_check.eval()

batch = next(iter(DataLoader([g_S, g_R, g_top], batch_size=3, shuffle=False))).to(device)
with torch.no_grad():
    out = model_check(batch.x, batch.edge_index, batch.edge_type, batch.batch, batch.protein_feat, batch.global_feat)
print(f"\n予測pIC50  S体: {out[0].item():.4f}  R体: {out[1].item():.4f}  差: {abs(out[0] - out[1]).item():.6f}")
print(f"確認 top_pred: {out[2].item():.4f}  ← val_predictions.csvの9.2910と一致することの確認")

c:\Users\todor\miniforge3\envs\bio_clean\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


smiles_to_graph: 学習データ先頭100件と一致
S体: CIP=[(16, 'S')]  Cc1sc2c(c1C)C(c1ccc(Cl)cc1)=N[C@@H](CC(=O)OC(C)(C)C)c1nnc(C)n1-2
R体: CIP=[(16, 'R')]  Cc1sc2c(c1C)C(c1ccc(Cl)cc1)=N[C@H](CC(=O)OC(C)(C)C)c1nnc(C)n1-2
SMILES文字列として同じか: False

x            完全一致: True
edge_index   完全一致: True
edge_type    完全一致: True
global_feat  完全一致: True

予測pIC50  S体: 7.0584  R体: 7.0584  差: 0.000000
確認 top_pred: 9.2910  ← val_predictions.csvの9.2910と一致することの確認


In [7]:
# ===== 5. DiffDockポーズの重心ズレ =====
# レポート5.3節。基準の取り方で結果が変わらないかを確かめるため、
# 結晶構造のJQ1と、DiffDockが予測した(S)-JQ1のrank1ポーズの両方を基準に計算する。
import glob
import numpy as np
from rdkit import Chem

DIFFDOCK_DIR = "." #このノートがあるフォルダ。DiffDockの実行ごとに diffdock_run○_○○ フォルダがある

def centroid(mol):
    return mol.GetConformer().GetPositions().mean(axis=0) #重原子の座標の平均(Colabの重心計算と同じ)

def load_rank1(run, name):
    path = glob.glob(f"{DIFFDOCK_DIR}/*/results/{run}/{name}/rank1_confidence*.sdf")
    assert len(path) == 1, f"{run}/{name} のrank1が{len(path)}件見つかりました(1件のはず)"
    return Chem.MolFromMolFile(path[0], removeHs=True, sanitize=False)

ref_crystal = Chem.MolFromPDBFile(f"{DIFFDOCK_DIR}/diffdock_run2_Sisomer/jq1_crystal.pdb", removeHs=True, sanitize=False) #結晶構造のJQ1
ref_pose = load_rank1("jq1_s_run", "JQ1_control_S") #DiffDockの(S)-JQ1 rank1
print(f"基準どうしの距離(結晶JQ1 vs (S)-JQ1 rank1): {np.linalg.norm(centroid(ref_crystal) - centroid(ref_pose)):.2f} Å  ← Colabノートの計算値(0.23 Å)と一致することの確認")

targets = [("top_pred", "brd4_run"), ("fp_highpred", "controls_run"), ("neg_true", "controls_run")]
print(f"\n{'化合物':<14}{'vs 結晶JQ1':>12}{'vs (S)-JQ1 rank1':>20}")
for name, run in targets:
    c = centroid(load_rank1(run, name))
    print(f"{name:<14}{np.linalg.norm(c - centroid(ref_crystal)):>10.2f} Å{np.linalg.norm(c - centroid(ref_pose)):>18.2f} Å")

基準どうしの距離(結晶JQ1 vs (S)-JQ1 rank1): 0.23 Å  ← Colabノートの計算値(0.23 Å)と一致することの確認

化合物               vs 結晶JQ1    vs (S)-JQ1 rank1
top_pred            0.92 Å              0.80 Å
fp_highpred         1.17 Å              1.21 Å
neg_true            3.32 Å              3.42 Å
